[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/rendering/examples/surfaces/qem/qem.ipynb)

# Quadric Error Metrics

This example simplifies a mesh by collapsing edges into vertices, placed where they disturb the surface least. Every triangle of the mesh lies in a plane, and moving a vertex off the planes of its triangles costs the sum of its squared distances to them. Collapsing an edge merges its two endpoints into one vertex that must serve the triangles of both, and the best place for it is where that summed cost is least: on a sharp ridge it stays on the ridge, pulled towards the sharper end.

In projective geometric algebra a plane is a vector and a point an antivector, and their pairing, `plane & point`, is the point's signed distance from a unit plane. A plane's dyad, `plane * (plane & Point)`, maps points to polar planes; paired back with the point, it gives squared distance. The error quadric of a vertex is the sum of its triangles' dyads. Collapsing an edge adds the endpoints' quadrics. A spatial correction from the edge's midpoint finds the minimum at unit homogeneous weight; when a plane or crease leaves the minimum undetermined, it selects the closest minimizer to that midpoint.

In the matrix notation of [Garland and Heckbert](https://www.cs.cmu.edu/~garland/quadrics/) each plane reads as a vector $\mathbf p$ of its four coefficients and its quadric as $\mathbf p\mathbf p^{\mathsf T}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga@rendering"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import concatenate, NumpyContext
from numga.algebras import PGA3D as ga
from examples import MODEL_DIR
from examples.util.animation import save_figure, show_animation
from examples.surfaces.qem import plumbing, render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()
Plane = ga.gatype.vector()
Quadric = ga.gatype((Plane, Point))
Spatial = ga.gatype.from_blades("x y z")
Direction = Spatial.dual().output_gatype

## A ridge, and its triangles' planes

The patch is six vertices and six triangles: a ridge from a sharp corner a to a gentler end b, two faces flanking the ridge edge, two steep faces at the corner and two shallow ones at the end. Each triangle's plane is the join of its three vertices.

In [ ]:
# The patch's vertices, a and b first, and its triangles, the two flanking the ridge edge first.
ridge = np.array([
    [ 0.35,  0.0,   0.25],  # 0: vertex a (sharp corner apex)
    [-0.35,  0.0,   0.25],  # 1: vertex b (crease continuation)
    [ 0.0,   0.55, -0.15],  # 2: left base vertex
    [ 0.0,  -0.55, -0.15],  # 3: right base vertex
    [ 0.75,  0.0,  -0.15],  # 4: front tip (steep corner at a)
    [-0.85,  0.0,  -0.15],  # 5: back tip (gentle ramp at b)
])
faces = np.array([
    [0, 1, 2],  # 0: left flank (shared by edge a-b)
    [1, 0, 3],  # 1: right flank (shared by edge a-b)
    [0, 2, 4],  # 2: corner front-left (incident to a only)
    [0, 4, 3],  # 3: corner front-right (incident to a only)
    [1, 5, 2],  # 4: ramp back-left (incident to b only)
    [1, 3, 5],  # 5: ramp back-right (incident to b only)
])
# The faces incident to each endpoint of the ridge edge (a, b), the two flanking faces in both; the faces
# that survive the collapse.
incident_a = np.array([0, 1, 2, 3])
incident_b = np.array([0, 1, 4, 5])
surviving = np.array([2, 3, 4, 5])

In [ ]:
vertices = mv.point(ridge)                                                     # [vertices] Point
face_planes = (vertices[faces[:, 0]] & vertices[faces[:, 1]] & vertices[faces[:, 2]]).normalized()   # [faces] Plane

In [ ]:
print("each face's plane against its three vertices:", np.abs((face_planes[:, None] & vertices[faces]).to_array()).max())

## Collapsing the edge

Each endpoint's quadric is the sum of its triangles' plane dyads, and the edge's quadric their sum. The best vertex has no spatial component in its polar plane. In the figure, left and middle, each endpoint with its triangles and the region where its error stays small; right, the collapsed mesh around the best vertex, green, with the original mesh faint behind it. The two flanking faces vanish with the edge.

In [ ]:
def collapse_position(quadric: Quadric, midpoint: Point) -> Point:
    """The unit-weight minimum, nearest the midpoint when the quadric has flat directions."""
    # A direction changes position without changing homogeneous weight. Only the spatial
    # part of the polar plane must vanish; its constant part is the residual minimum error.
    correction = quadric(Direction).cast(Spatial).lstsq(-quadric(midpoint).cast(Spatial))
    return midpoint + correction


# A unit plane measures a point's signed distance from it as `plane & point`; its dyad, a map from points to planes,
# gives the squared distance. Each endpoint's error quadric sums the dyads of its triangles' planes.
dyads = face_planes * (face_planes & Point)                                    # [faces] Quadric
quadric_a, quadric_b = dyads[incident_a].sum(axis=0), dyads[incident_b].sum(axis=0)   # Quadric each
# Contracting the edge adds the endpoints' quadrics; the best vertex is its unit-weight minimum.
edge_quadric = quadric_a + quadric_b                                           # Quadric
optimum = collapse_position(edge_quadric, (vertices[0] + vertices[1]) / 2)     # Point
collapsed = concatenate([optimum.broadcast_to((2,)), vertices[2:]])           # [vertices] Point
save_figure(render.draw_qem(vertices, faces, incident_a, incident_b, quadric_a, quadric_b, edge_quadric, optimum,
                            collapsed, surviving), "qem_mesh_simplification");

In [ ]:
endpoints = vertices[:2]
print("the best vertex:\n", optimum.coefficient_table())
print("its error, against the endpoints':", (edge_quadric(optimum) & optimum).to_array(), (edge_quadric(endpoints) & endpoints).to_array())

In [ ]:
# checks
# The best vertex's polar plane is the plane at infinity; it stays on the ridge, between its middle and
# the sharp corner, and its error undercuts both endpoints'.
assert edge_quadric(optimum).norm().to_array() < 1e-9
np.testing.assert_allclose((mv.y & optimum).to_array(), 0.0, atol=1e-12)
assert 0.0 < (mv.x & optimum).to_array() < (mv.x & vertices[0]).to_array()
assert np.all((edge_quadric(optimum) & optimum).to_array() < (edge_quadric(endpoints) & endpoints).to_array())

## Simplifying the bunny

The same construction reduces the bunny from 6,966 triangles to 250. Each merged vertex keeps the sum of its predecessors' quadrics, so the error continues to refer to the original surface. Each pass chooses low-error contractions in separate neighbourhoods, keeping the mesh closed and its faces facing the same way. The camera stays fixed; flat shading makes the surviving facets visible, especially around the ears and silhouette.

In [ ]:
def simplify(vertices: Point, faces: np.ndarray, targets: np.ndarray) -> Iterator[tuple[Point, np.ndarray]]:
    """A closed triangle mesh, then its QEM contractions at descending face-count targets.

    The initial face planes remain the reference surface: every merged vertex retains the
    sum of its original vertices' quadrics. Each pass contracts separate neighbourhoods,
    choosing the least errors first and preserving the topology and the sense of each face.
    """
    corners = vertices[faces]
    planes = (corners[:, 0] & corners[:, 1] & corners[:, 2]).normalized()
    face_quadrics = planes * (planes & Point)
    quadrics = (face_quadrics[0] * 0).broadcast_to((len(vertices),))
    quadrics = quadrics.at[faces.reshape(-1)].add(face_quadrics[:, None].broadcast_to((len(faces), 3)).reshape(-1))
    yield vertices, faces

    for target in targets:
        while len(faces) > target:
            topology = plumbing.Topology(faces, len(vertices))
            first, second = topology.edges.T
            edge_quadrics = quadrics[first] + quadrics[second]
            midpoints = (vertices[first] + vertices[second]) / 2
            optima = collapse_position(edge_quadrics, midpoints)
            errors = edge_quadrics(optima) & optima

            # Test every surviving triangle in an edge's neighbourhood with that edge
            # contracted. Its plane must keep its sense and a nonzero area.
            before = vertices[faces[topology.trial_faces]]
            after = before.at[np.arange(len(before)), topology.trial_corners].set(optima[topology.trial_edges])
            old_planes = before[:, 0] & before[:, 1] & before[:, 2]
            new_planes = after[:, 0] & after[:, 1] & after[:, 2]
            consistent = (old_planes.scalar_product(new_planes) > 0) & (new_planes.squared() > old_planes.squared() * 1e-12)
            selected = plumbing.select_edges(topology, errors, consistent, max(1, (len(faces) - target) // 2))
            if not len(selected):
                raise ValueError(f"No legal contraction remains at {len(faces)} faces before target {target}.")

            # The selected stars do not share changed faces, so their independent
            # contractions can be applied together without invalidating the face tests.
            vertices = vertices.at[first[selected]].set(optima[selected])
            quadrics = quadrics.at[first[selected]].set(edge_quadrics[selected])
            retained, faces = plumbing.contract(topology, selected)
            vertices, quadrics = vertices[retained], quadrics[retained]
        yield vertices, faces

In [ ]:
mesh_size = 2
face_targets = np.array([3500, 1800, 900, 450, 250])
duration_ms = 600
vertices, faces = plumbing.read_mesh(MODEL_DIR / "bunny.obj", mesh_size)
# The model's vertical axis is y; stand it upright along z.
upright = (mv.yz * (-np.pi / 4)).exp()
vertices = upright >> vertices

states = simplify(vertices, faces, face_targets)
show_animation(render.animate_mesh(states, vertices), "qem_bunny", duration_ms,
               n_frames=len(face_targets) + 1)